<div>
<hr style="width: 80%;">

<p align="center" style="font-size: 20px; color: #1f4e79; margin: 0;"><strong>Sistema de Reconocimiento de Posturas de Yoga</strong></p>
<p align="center" style="font-size: 16px; margin: 0;"><strong>Notebook de entrenamiento</strong> — se ejecuta en Google Colab (GPU)</p>

<p style="font-size: 15px;">Este notebook parte de la actividad académica original
(<em>"Evaluemos un modelo de clasificación"</em>, Maestría en IA — Visión por Computador) y la
convierte en el pipeline de entrenamiento del modelo que usa la aplicación
<strong>Sistema de Reconocimiento de Posturas de Yoga</strong> (carpetas <code>backend/</code> y
<code>frontend/</code> de este mismo repositorio).</p>

<p style="font-size: 15px;"><strong>Salida de este notebook</strong> (se descargan de Colab y se colocan en
<code>backend/models/</code>):</p>
<ul style="font-size: 15px;">
<li><code>modelo_yoga_kaggle.h5</code> — modelo entrenado</li>
<li><code>labels.json</code> — lista ordenada de clases (el backend no debe asumir el orden)</li>
<li><code>model_metrics.json</code> — métricas de evaluación sobre el conjunto de TEST real</li>
</ul>
<hr style="width: 80%;">
</div>

#### **Cambios respecto al notebook original**

1. Se corrige el orden de `.ignore_errors()`: SI es un metodo valido de `tf.data.Dataset` (descarta elementos que fallan al decodificarse) y es necesario porque el dataset trae al menos un JPEG corrupto -- confirmado en la practica, sin esto el entrenamiento se cae a mitad de una epoca. Se aplica justo despues de crear cada dataset (antes de `cache`), en vez de al final de la cadena.
2. Se agrega **data augmentation** (flip, rotacion, zoom, contraste) para reducir sobreajuste en un dataset pequeno.
3. Se reemplazan las 100 epocas fijas por **EarlyStopping + ModelCheckpoint + ReduceLROnPlateau**.
4. Se agrega una segunda etapa de **fine-tuning** (descongelar las ultimas capas de MobileNetV2) para intentar mejorar sobre el ~78% de accuracy original.
5. La evaluacion final (classification report, matriz de confusion, curvas ROC) se hace sobre el conjunto **TEST real** (antes se reusaba el split de validacion, lo cual mezclaba seleccion de modelo con evaluacion final).
6. `class_names` se deriva una sola vez y se exporta a `labels.json` en vez de hardcodearse una segunda vez.
7. Se exportan artefactos (`labels.json`, `model_metrics.json`) listos para el backend.

**Resultado real obtenido** entrenando este notebook (ver `training/train_local.py` para la version ejecutada localmente): **78.3% de accuracy** sobre el conjunto de TEST real, con `goddess` como la clase mas dificil (F1 0.66) y `downdog`/`plank` como las mejor reconocidas -- consistente con lo observado en el notebook academico original.

####**Librerías necesarias**

------

En este paso cargamos las herramientas que usaremos para construir, entrenar y evaluar el modelo (TensorFlow, Keras, scikit-learn y Matplotlib).

In [ ]:
# Importar librerías
import os
import json
import itertools

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
    accuracy_score,
)
from sklearn.preprocessing import label_binarize

###**Paso 1: Descargar el dataset desde KaggleHub**

------

En este paso importamos el conjunto de imágenes directamente desde Kaggle usando KaggleHub.
Así evitamos subir archivos manualmente y tenemos acceso rápido y ordenado a las carpetas del dataset de posturas de yoga.

In [ ]:
import kagglehub
import shutil

# 1. Descargar el dataset desde Kaggle
print("Descargando dataset desde Kaggle...")
path = kagglehub.dataset_download("niharika41298/yoga-poses-dataset")
print("Descarga completada.")
print("Ruta original del dataset:", path)

# 2. Crear carpeta visible en /content
destino = "/content/yoga_poses_dataset"

# Si ya existe, la borramos
if os.path.exists(destino):
    shutil.rmtree(destino)

# Copiamos todo el contenido descargado a /content
shutil.copytree(path, destino)

print("Dataset copiado a:", destino)

###**Paso 2: Verificar la estructura del dataset**

------

En este paso revisamos cómo está organizado el conjunto de imágenes descargado.
Listamos las carpetas principales y exploramos una de ellas para asegurarnos de que las fotos de cada postura (por ejemplo, tree, warrior2, downdog) estén bien clasificadas y listas para usar en el entrenamiento del modelo.

In [ ]:
# Listar las carpetas principales
for folder in os.listdir(path):
    print("-", folder)

# Accedemos a la subcarpeta principal (por ejemplo, "Yoga-Dataset")
subfolder = os.path.join(path, os.listdir(path)[0])
print("\nExplorando dentro de:", subfolder)
print(os.listdir(subfolder)[:10])

###**Paso 3. Definir rutas del dataset**

------

En este paso indicamos dónde están guardadas las imágenes de entrenamiento y prueba dentro del conjunto descargado.

In [ ]:
# Definir rutas de TRAIN y TEST de forma dinámica
train_path = os.path.join(subfolder, "TRAIN")
test_path = os.path.join(subfolder, "TEST")

print("\nCarpetas dentro de TRAIN:")
print(os.listdir(train_path))

print("\nCarpetas dentro de TEST:")
print(os.listdir(test_path))

####**Paso 4. Crear datasets base (entrenamiento y validación)**

------

Cargamos las imágenes de `TRAIN` y las dividimos: 80% para entrenar el modelo y 20% para validar qué tan bien está aprendiendo durante el entrenamiento. Todas las imágenes se redimensionan a 224×224 píxeles y se agrupan en lotes de 32.

In [ ]:
train_dir = train_path

# 1. Conjunto de entrenamiento (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
)

# 2. Conjunto de validación (20% del TRAIN, usado durante el entrenamiento)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
)

# 3. Clases detectadas (se deriva UNA sola vez; el resto del notebook reutiliza esta lista)
class_names = train_ds.class_names
print("Clases detectadas:", class_names)

####**Paso 4.1 Crear dataset de prueba (TEST real)**

------

*(Nuevo respecto al notebook original)* Cargamos también la carpeta `TEST`, que el modelo no ve durante el entrenamiento ni la selección de hiperparámetros. Es el conjunto que usaremos más adelante para la evaluación final — así evitamos medir el modelo con los mismos datos que ayudaron a elegirlo.

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    shuffle=False,
    seed=123,
    image_size=(224, 224),
    batch_size=32,
)

assert test_ds.class_names == class_names, (
    "El orden de clases de TEST no coincide con el de TRAIN: "
    f"{test_ds.class_names} vs {class_names}"
)

####**Paso 5. Optimizar rendimiento de lectura**

------

Mejoramos la velocidad con la que el modelo lee las imágenes durante el entrenamiento.

In [ ]:
# El dataset trae al menos un JPEG corrupto (confirmado en la practica: rompe la
# decodificacion a mitad de entrenamiento). ignore_errors() SI es un metodo valido de
# tf.data.Dataset -- descarta silenciosamente los elementos que fallan al decodificarse.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.ignore_errors().cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.ignore_errors().cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.ignore_errors().cache().prefetch(buffer_size=AUTOTUNE)

####**Paso 5.1 Aumento de datos (data augmentation)**

------

*(Nuevo respecto al notebook original)* El dataset es pequeño, así que aplicamos transformaciones aleatorias (volteo horizontal, rotación, zoom y contraste) a las imágenes de entrenamiento en cada época. Esto ayuda a que el modelo generalice mejor en vez de memorizar las fotos exactas.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ],
    name="data_augmentation",
)

####**Paso 6. Modelo base preentrenado (MobileNetV2)**

------

Utilizamos un modelo ya entrenado llamado MobileNetV2, que ha aprendido a reconocer miles de objetos en el conjunto de datos ImageNet. Aprovechamos ese conocimiento previo para que nuestro modelo no empiece desde cero.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Congelamos los pesos base en esta primera etapa

**Importante:**

Lo que se hizo fue traer un modelo ya conocido llamado MobileNetV2, pero solo su parte base, la que sabe reconocer formas, colores y texturas gracias a haber sido entrenado con millones de fotos (el conjunto ImageNet), pero todavía no sabe identificar las posturas de yoga, porque no ha aprendido las clases específicas (tree, plank, warrior2, etc.).

####**Paso 7. Construir y entrenar el modelo (etapa 1: solo las capas nuevas)**

------

Construimos la parte final del modelo: primero la capa de aumento de datos, luego la base MobileNetV2, después `GlobalAveragePooling2D` para resumir la información visual, `Dropout` para evitar sobreajuste y finalmente una capa de salida con una neurona por postura.

En vez de forzar un número fijo de épocas, usamos **EarlyStopping** (detiene el entrenamiento si deja de mejorar), **ModelCheckpoint** (guarda el mejor modelo visto) y **ReduceLROnPlateau** (reduce la tasa de aprendizaje si el modelo se estanca).

In [ ]:
# 1. Crear el modelo final (funcional, para poder reusarlo en la etapa de fine-tuning)
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# 2. Compilar el modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

# 3. Callbacks: paran el entrenamiento cuando deja de mejorar y guardan el mejor modelo
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/modelo_yoga_kaggle_best.h5', monitor='val_accuracy', save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
    ),
]

# 4. Entrenar (tope de 30 épocas; EarlyStopping normalmente detiene antes)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

####**Paso 7.1 Fine-tuning (etapa 2: afinar las últimas capas de MobileNetV2)**

------

*(Nuevo respecto al notebook original)* Ahora descongelamos las últimas capas de MobileNetV2 y seguimos entrenando con una tasa de aprendizaje muy baja. Esto permite que la base preentrenada se ajuste ligeramente a las texturas y formas específicas de las posturas de yoga, en vez de quedarse fija con lo aprendido en ImageNet.

In [ ]:
base_model.trainable = True

# Solo descongelamos las últimas 30 capas; el resto de la base sigue congelada
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

fine_tune_epochs = 15
total_epochs = history.epoch[-1] + 1 + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=callbacks,
)

####**Paso 7.2 Guardar los artefactos del modelo**

------

Guardamos el modelo entrenado y la lista de clases. El backend de la aplicación carga estos dos archivos (más `model_metrics.json`, generado al final del notebook) — **no** debe asumir el orden de clases, por eso se exporta `labels.json` explícitamente.

In [ ]:
model.save('/content/modelo_yoga_kaggle.h5')

with open('/content/labels.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print('Modelo guardado como modelo_yoga_kaggle.h5')
print('Clases guardadas en labels.json:', class_names)

###**Paso 8. Análisis y clasificación de una imagen de ejemplo**

------

Tomamos una imagen del conjunto de TEST para observar cómo el modelo la interpreta. Primero un pequeño análisis estadístico (brillo, contraste, colores predominantes) y luego la predicción del modelo entrenado.

In [ ]:
# Tomamos la primera imagen disponible de la clase 'downdog' dentro de TEST
downdog_dir = os.path.join(test_path, 'downdog')
sample_file = sorted(os.listdir(downdog_dir))[0]
img_path = os.path.join(downdog_dir, sample_file)

img = load_img(img_path, target_size=(224, 224))
img_array = img_to_array(img) / 255.0

plt.figure(figsize=(4, 4))
plt.imshow(img_array)
plt.axis('off')
plt.title('Imagen seleccionada para el análisis')
plt.show()

# Estadísticas básicas
valor_minimo = np.min(img_array)
valor_maximo = np.max(img_array)
promedio = np.mean(img_array)
desviacion = np.std(img_array)

print('Estadísticas básicas de la imagen:\n')
print(f'- Dimensiones: {img_array.shape} (alto, ancho, canales de color)')
print(f'- Valor mínimo: {valor_minimo:.3f}')
print(f'- Valor máximo: {valor_maximo:.3f}')
print(f'- Promedio general (brillo): {promedio:.3f}')
print(f'- Desviación estándar (contraste): {desviacion:.3f}')

In [ ]:
# Gráfico de torta — proporción promedio de color
mean_colors = np.mean(img_array, axis=(0, 1))

plt.figure(figsize=(4.5, 4.5))
plt.pie(
    mean_colors,
    labels=['Rojo', 'Verde', 'Azul'],
    colors=['red', 'green', 'blue'],
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
)
plt.title('Proporción promedio de color en la imagen')
plt.tight_layout()
plt.show()

In [ ]:
# Predicción de la postura utilizando el modelo entrenado
img_ready = np.expand_dims(img_array, axis=0)
pred = model.predict(img_ready, verbose=0)
predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred)

print('Resultado de la predicción:')
print(f'- Postura detectada: {predicted_class}')
print(f'- Nivel de confianza: {confidence:.2f}')

plt.figure(figsize=(4, 4))
plt.imshow(img_array)
plt.axis('off')
plt.title(f'Postura: {predicted_class} ({confidence:.2f})')
plt.show()

###**Paso 9. Evaluación del modelo sobre el conjunto de TEST**

------

*(Corregido respecto al notebook original: aquí se evalúa sobre `test_ds`, el conjunto de prueba real, no sobre el split de validación que ya se usó para decidir cuándo detener el entrenamiento.)*

Analizamos tres indicadores principales — precisión, recall y F1-score — y visualizamos la matriz de confusión y las curvas ROC.

In [ ]:
y_true, y_pred, y_prob = [], [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print('--- Evaluación del modelo sobre el conjunto de TEST ---\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de confusión - Conjunto de TEST')
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(
        j, i, format(cm[i, j], 'd'),
        ha='center', va='center',
        color='white' if cm[i, j] > thresh else 'black',
    )

plt.ylabel('Postura real')
plt.xlabel('Postura predicha')
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC por clase
y_true_bin = label_binarize(y_true, classes=np.arange(len(class_names)))

plt.figure(figsize=(7, 6))
for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Modelo aleatorio')
plt.title('Curvas ROC por clase - Conjunto de TEST')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

####**Paso 9.1 Exportar métricas para la aplicación**

------

*(Nuevo respecto al notebook original)* Guardamos las métricas en `model_metrics.json` para que la aplicación pueda mostrar información sobre el desempeño del modelo (por ejemplo en una sección "Acerca del modelo").

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(len(class_names))
)
overall_accuracy = accuracy_score(y_true, y_pred)

metrics = {
    'accuracy': float(overall_accuracy),
    'per_class': {
        class_names[i]: {
            'precision': float(precision[i]),
            'recall': float(recall[i]),
            'f1_score': float(f1[i]),
            'support': int(support[i]),
        }
        for i in range(len(class_names))
    },
}

with open('/content/model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Métricas exportadas a model_metrics.json')
print(json.dumps(metrics, indent=2, ensure_ascii=False))

###**Siguiente paso**

------

Descarga de la carpeta `/content/` de Colab estos 3 archivos:

- `modelo_yoga_kaggle.h5`
- `labels.json`
- `model_metrics.json`

y colócalos en `backend/models/` de este repositorio. Con eso, la API (`backend/`) queda lista para servir predicciones reales — instrucciones detalladas en `training/README.md` y `backend/README.md`.